In [0]:
silver_path = "/Volumes/hr_project_catalog/hr_schema/hr_project_volume/silver/"

In [0]:
from pyspark.sql.functions import col

df_emp_clean = spark.read.parquet(
    "/Volumes/hr_project_catalog/hr_schema/hr_project_volume/bronze/employees"
).withColumn("join_date", col("join_date").cast("Date"))

display(df_emp_clean)

In [0]:
from pyspark.sql.functions import year, when, col

df_emp_clean = df_emp_clean.withColumn("join_year", year(col("join_date")))

df_emp_clean = df_emp_clean.withColumn(
    "experience_level", 
    when(col("join_year") < 2019, "Senior")
    .when((col("join_year") >= 2019) & (col("join_year") <= 2021), "Mid")
    .otherwise("Junior")
).drop("join_year")  

display(df_emp_clean)

In [0]:
from pyspark.sql.functions import col, datediff, lit, to_date, current_timestamp

df_emp_clean = df_emp_clean.withColumn("days_employed", datediff(to_date(current_timestamp(), "yyyy-MM-dd"), col("join_date")))

display(df_emp_clean)

In [0]:
from pyspark.sql.functions import year, when, col, datediff, lit, to_date, current_timestamp

df_emp_clean = df_emp_clean.withColumn(
    "salary_band",
    when(col("salary") >= 90000, "High")
    .when(col("salary") >= 70000, "Mid")
    .otherwise("Low")
)

display(df_emp_clean)

In [0]:
df_emp_clean = df_emp_clean.filter(col("is_active")== "True")

display(df_emp_clean)

In [0]:
df_emp_clean = df_emp_clean.withColumnRenamed("dept", "department")
df_emp_clean = df_emp_clean.drop("gender")

display(df_emp_clean)

In [0]:
df_emp_clean.write.mode("overwrite").parquet(silver_path + "employees_clean")